# Automatic Classification of LoRaWAN Environments
### Complete ML pipeline — 7 labeled files
---
**Files:**

| File | Environment label |
|---|---|
| Gateway_Data_LOS_Apartment | Residential LOS |
| Gateway_Data_NLOS_Apartment | Residential NLOS |
| Gateway_Data_LOS_Labo | Laboratory LOS |
| Gateway_Data_NLOS_Labo | Laboratory NLOS |
| Gateway_Data_EMSI | Multi-floor |
| Gateway_Data_UP_Scenario1 | Industrial Scen.1 |
| Gateway_Data_UP_Scenario2 | Industrial Scen.2 |

**Pipeline:** Feature engineering → PCA/t-SNE → RF / SVM / XGBoost / KNN → Confusion matrix → ROC → Feature importance


In [ ]:
import warnings, os, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec

# Preprocessing & evaluation
from sklearn.preprocessing import StandardScaler, LabelEncoder, label_binarize
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.model_selection import (StratifiedKFold, cross_val_score,
                                     train_test_split, GridSearchCV)
from sklearn.metrics import silhouette_score
from sklearn.metrics import (classification_report, confusion_matrix,
                              roc_curve, auc, ConfusionMatrixDisplay)
from sklearn.inspection import permutation_importance
from sklearn.pipeline import Pipeline

# -- Family 1: Linear methods --
from sklearn.linear_model import (LogisticRegression, RidgeClassifier,
                                   SGDClassifier, Perceptron)

# -- Family 2: Decision trees --
from sklearn.tree import DecisionTreeClassifier, ExtraTreeClassifier

# -- Family 3: Ensembles (Bagging / Boosting / Stacking) --
from sklearn.ensemble import (RandomForestClassifier, ExtraTreesClassifier,
                               GradientBoostingClassifier, AdaBoostClassifier,
                               BaggingClassifier, StackingClassifier,
                               VotingClassifier)

# -- Family 4: Kernel methods --
from sklearn.svm import SVC, LinearSVC, NuSVC

# -- Family 5: Instance-based --
from sklearn.neighbors import KNeighborsClassifier, RadiusNeighborsClassifier

# -- Family 6: Probabilistic / Bayesian --
from sklearn.naive_bayes import GaussianNB, BernoulliNB, ComplementNB

# -- Family 7: Neural networks --
from sklearn.neural_network import MLPClassifier

# -- Family 8: Discriminant Analysis --
from sklearn.discriminant_analysis import (LinearDiscriminantAnalysis,
                                            QuadraticDiscriminantAnalysis)

# -- XGBoost (optional) --
try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print("[!] XGBoost not found -- pip install xgboost")

try:
    from lightgbm import LGBMClassifier
    HAS_LGB = True
except ImportError:
    HAS_LGB = False
    print("[!] LightGBM not found -- pip install lightgbm")

try:
    from catboost import CatBoostClassifier
    HAS_CAT = True
except ImportError:
    HAS_CAT = False
    print("[!] CatBoost not found -- pip install catboost")

warnings.filterwarnings("ignore")
plt.rcParams.update({
    "figure.dpi": 150, "figure.facecolor": "white", "axes.facecolor": "white",
    "axes.grid": True, "grid.alpha": 0.3, "grid.linestyle": "--",
    "axes.spines.top": False, "axes.spines.right": False,
    "font.size": 10, "axes.titlesize": 11, "axes.titleweight": "bold",
})

OUTPUT_DIR = "results_ML"
os.makedirs(OUTPUT_DIR, exist_ok=True)
RANDOM_STATE = 42
CV_FOLDS = 5
print(f"Imports OK | XGBoost:{HAS_XGB} | LightGBM:{HAS_LGB} | CatBoost:{HAS_CAT}")


In [ ]:
# ================================================================
#  CONFIGURATION -- 4 environments (grouped files) or 7 raw classes
# ================================================================
#
# Paths are relative to this notebook's location (Machine Learning-Based
# Environment Classification/), assuming the repository layout:
#   repo_root/
#   |-- Dataset/
#   |   \-- By_Environment_and_Condition/   <- CSV files referenced below
#   \-- Machine Learning-Based Environment Classification/   <- this notebook
#
# File names match the deposited dataset (see
# ../Dataset/By_Environment_and_Condition/README.md).

DATA_DIR = "../Dataset/By_Environment_and_Condition/"

# Set to "7-class" to keep LOS/NLOS/Scenario sub-conditions as separate
# classes instead of merging them into the 4 paper-level environments.
# Useful to check whether misclassifications are mostly within the same
# physical site (LOS <-> NLOS) or across genuinely different environments.
CLASSIFICATION_MODE = "4-class"   # "4-class" or "7-class"

FILES_4CLASS = {
    "Residential":  [DATA_DIR + "Residential_LOS.csv",
                      DATA_DIR + "Residential_NLOS.csv"],
    "Laboratory":   [DATA_DIR + "Laboratory_LOS.csv",
                      DATA_DIR + "Laboratory_NLOS.csv"],
    "Multi-floor":  [DATA_DIR + "MultiFloorBuilding_NLOS.csv"],
    "Industrial":   [DATA_DIR + "UndergroundIndustrial_Scenario1.csv",
                      DATA_DIR + "UndergroundIndustrial_Scenario2.csv"],
}

FILES_7CLASS = {
    "Residential_LOS":     [DATA_DIR + "Residential_LOS.csv"],
    "Residential_NLOS":    [DATA_DIR + "Residential_NLOS.csv"],
    "Laboratory_LOS":      [DATA_DIR + "Laboratory_LOS.csv"],
    "Laboratory_NLOS":     [DATA_DIR + "Laboratory_NLOS.csv"],
    "Multi-floor_NLOS":    [DATA_DIR + "MultiFloorBuilding_NLOS.csv"],
    "Industrial_Scenario1": [DATA_DIR + "UndergroundIndustrial_Scenario1.csv"],
    "Industrial_Scenario2": [DATA_DIR + "UndergroundIndustrial_Scenario2.csv"],
}

PALETTE_4CLASS = {
    "Residential":  "#1D9E75",
    "Laboratory":   "#378ADD",
    "Multi-floor":  "#D85A30",
    "Industrial":   "#7F77DD",
}

PALETTE_7CLASS = {
    "Residential_LOS":      "#1D9E75",
    "Residential_NLOS":     "#7FD4B8",
    "Laboratory_LOS":       "#378ADD",
    "Laboratory_NLOS":      "#A8CDEF",
    "Multi-floor_NLOS":     "#D85A30",
    "Industrial_Scenario1": "#7F77DD",
    "Industrial_Scenario2": "#C2BEEF",
}

if CLASSIFICATION_MODE == "7-class":
    FILES   = FILES_7CLASS
    PALETTE = PALETTE_7CLASS
else:
    FILES   = FILES_4CLASS
    PALETTE = PALETTE_4CLASS

print(f"Classification mode: {CLASSIFICATION_MODE} ({len(FILES)} classes)")


## Step 1 -- Loading and feature engineering


In [ ]:
def load_file(filepath, label):
    """Load a CSV/TSV file, detect the delimiter, clean the Airtime column."""
    with open(filepath, "r", encoding="utf-8", errors="replace") as f:
        first = f.readline()
    tabs = first.count("\t"); commas = first.count(","); semis = first.count(";")
    sep = "\t" if tabs >= commas and tabs >= semis else (";" if semis > commas else ",")

    df = pd.read_csv(filepath, sep=sep, na_values=["NaN","nan","","N/A"], engine="python")

    # Clean Airtime
    df["Airtime_s"] = (df["Airtime_s"].astype(str)
                       .str.replace("s", "", regex=False).str.strip())
    df["Airtime_s"] = pd.to_numeric(df["Airtime_s"], errors="coerce")

    # Optional columns
    for col in ["FrequencyOffset_Hz", "ChannelRSSI_dBm", "ChannelIndex", "Conditions_1"]:
        if col not in df.columns:
            df[col] = np.nan

    df["environment"] = label
    return df


def build_features(df):
    """Build the per-packet feature vector.

    Feature definitions and rationale:
      - rssi, snr, sf, freq_mhz, airtime_ms, freq_offset: raw telemetry
        fields, used directly.
      - rssi_delta = RSSI_dBm - ChannelRSSI_dBm: the magnitude of the
        SNR-based RSSI adjustment applied by the LoRaWAN network server
        (RSSI_dBm is ChannelRSSI_dBm corrected for SNR, following the
        standard LoRaWAN convention -- see data/README.md). This is
        near-zero whenever SNR is comfortably positive (the common case
        for Residential/Laboratory) and grows in magnitude as SNR becomes
        negative, so it is retained as a physically meaningful, well
        -defined feature despite showing low permutation importance in
        practice.
      - rssi_snr_ratio = RSSI_dBm / (SNR_dB + 20): an engineered
        interaction term (not a physical law) intended to let tree-based
        models access a simple nonlinear combination of RSSI and SNR
        without an explicit multiplicative split; retained because it
        shows non-negligible permutation importance.
      - snr_positive: binary indicator of whether SNR is above the
        decoding-threshold-adjacent zero point, capturing the regime
        change described in Section 4.1.3.

    Note: an `rssi_abs = abs(RSSI_dBm)` feature was considered but is not
    included here. Since RSSI is always negative in this dataset,
    abs(RSSI_dBm) is a strictly monotonic transform of RSSI_dBm; for
    tree-based classifiers, a split on abs(RSSI) < threshold is
    equivalent to a split on RSSI > -threshold, so this feature cannot
    carry information beyond what RSSI_dBm alone already provides. It
    was previously included and consistently showed zero permutation
    importance, consistent with this reasoning.
    """
    f = pd.DataFrame()
    f["rssi"]           = df["RSSI_dBm"]
    f["snr"]            = df["SNR_dB"]
    f["sf"]             = df["SpreadingFactor_SF"]
    f["freq_mhz"]       = df["Frequency_Hz"] / 1e6
    f["airtime_ms"]     = df["Airtime_s"] * 1000
    f["freq_offset"]    = df["FrequencyOffset_Hz"]
    f["rssi_delta"]     = df["RSSI_dBm"] - df["ChannelRSSI_dBm"]
    f["rssi_snr_ratio"] = df["RSSI_dBm"] / (df["SNR_dB"] + 20)
    f["snr_positive"]   = (df["SNR_dB"] > 0).astype(int)
    if df["Conditions_1"].dtype in [np.float64, np.int64]:
        f["conditions_1"] = df["Conditions_1"]
    return f


# -- Load and group into 4 environments --
frames = []
print(f"{'File':<42} {'Environment':<16} {'Packets':>8}")
print("-" * 68)

for label, paths in FILES.items():
    # Accept either a list or a single string
    if isinstance(paths, str):
        paths = [paths]
    for path in paths:
        try:
            d = load_file(path, label)
            frames.append(d)
            print(f"  {path:<40} {label:<16} {len(d):>6}")
        except FileNotFoundError:
            print(f"  [x] {path} -- not found")

if not frames:
    raise RuntimeError("No file loaded. Check the paths in FILES.")

df_all = pd.concat(frames, ignore_index=True)
print(f"\nTotal: {len(df_all)} packets | {df_all['environment'].nunique()} classes")
print("\nDistribution by environment:")
print(df_all["environment"].value_counts().to_string())


In [ ]:
# -- Build the ML dataset --
X_raw = build_features(df_all)
FEATURE_NAMES = X_raw.columns.tolist()

# Median imputation
X_raw = X_raw.fillna(X_raw.median(numeric_only=True))

# Label encoding
le = LabelEncoder()
y  = le.fit_transform(df_all["environment"])

# Standardization
scaler  = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

print(f"Features ({len(FEATURE_NAMES)}) : {FEATURE_NAMES}")
print(f"Classes  ({len(le.classes_)})  : {list(le.classes_)}")


## Step 1b -- Formal significance test on RSSI and SNR across environments (Section 4.1.3)

In [ ]:
# One-way ANOVA and its non-parametric analogue (Kruskal-Wallis) on RSSI
# and SNR across the four environments, complementing the Wilson-interval
# PDR comparison with a direct test on the two dominant physical-layer
# metrics themselves. A Shapiro-Wilk check (on a capped sample, since the
# test is undefined for very large n) reports whether each group is close
# enough to normal for the ANOVA result to be trusted at face value.
from scipy.stats import f_oneway, kruskal, shapiro

rng_shapiro = np.random.default_rng(RANDOM_STATE)  # fixed seed for the Shapiro-Wilk subsample
anova_rows = []
for metric_col, metric_name in [("RSSI_dBm", "RSSI"), ("SNR_dB", "SNR")]:
    groups = [g[metric_col].dropna().values for _, g in df_all.groupby("environment")]

    f_stat, p_anova = f_oneway(*groups)
    h_stat, p_kw = kruskal(*groups)

    normal_flags = []
    for g in groups:
        sample = g if len(g) <= 5000 else rng_shapiro.choice(g, 5000, replace=False)
        _, p_norm = shapiro(sample)
        normal_flags.append(p_norm > 0.05)

    anova_rows.append({
        "Metric": metric_name,
        "ANOVA_F": round(f_stat, 2), "ANOVA_p": p_anova,
        "KruskalWallis_H": round(h_stat, 2), "KruskalWallis_p": p_kw,
        "All_groups_normal_ShapiroWilk": all(normal_flags),
        "N_groups": len(groups),
    })

anova_df = pd.DataFrame(anova_rows)
anova_df.to_csv(f"{OUTPUT_DIR}/Table_ANOVA_RSSI_SNR.csv", index=False)

print(f"{'='*70}")
print("  RSSI/SNR across environments -- ANOVA and Kruskal-Wallis")
print(f"{'='*70}")
for _, r in anova_df.iterrows():
    print(f"  {r['Metric']:<6} ANOVA F={r['ANOVA_F']:>10.1f} (p={r['ANOVA_p']:.3g})   "
          f"Kruskal-Wallis H={r['KruskalWallis_H']:>8.1f} (p={r['KruskalWallis_p']:.3g})   "
          f"normal={r['All_groups_normal_ShapiroWilk']}")
print(f"{'='*70}")
print("Note: normality is rejected in at least one group for both metrics, "
      "so the Kruskal-Wallis result is the more appropriate of the two; "
      "both point in the same direction here.")
display(anova_df)


## Step 1c -- Packet distribution per node and over time (Section 3.2)

In [ ]:
# Breakdown of the dataset by physical node and by collection day, beyond
# the aggregate per-environment counts already shown above.
node_counts = (df_all.groupby(["environment", "Device_ID"])
               .size().reset_index(name="Packets")
               .sort_values(["environment", "Packets"], ascending=[True, False]))
node_counts.to_csv(f"{OUTPUT_DIR}/Table_PacketsPerNode.csv", index=False)

print(f"{'='*60}")
print(f"  {len(node_counts)} distinct (environment, node) groups -- "
      f"first 10 rows shown, full table saved to CSV")
print(f"{'='*60}")
display(node_counts.head(10))

if "DR_Time" in df_all.columns:
    try:
        ts = pd.to_datetime(df_all["DR_Time"].astype(float), unit="s", errors="coerce")
    except (ValueError, TypeError):
        ts = pd.to_datetime(df_all["DR_Time"], errors="coerce")

    if ts.notna().any():
        df_all["_parsed_date"] = ts.dt.date
        daily_counts = (df_all.dropna(subset=["_parsed_date"])
                        .groupby(["environment", "_parsed_date"])
                        .size().reset_index(name="Packets"))
        daily_counts.to_csv(f"{OUTPUT_DIR}/Table_PacketsPerDay.csv", index=False)

        span = (df_all.dropna(subset=["_parsed_date"])
                .groupby("environment")["_parsed_date"]
                .agg(["min", "max", "nunique"])
                .rename(columns={"min": "First_day", "max": "Last_day",
                                 "nunique": "N_distinct_days"}))
        span.to_csv(f"{OUTPUT_DIR}/Table_TemporalSpan.csv")
        print(f"\n{'='*60}\n  Temporal span per environment\n{'='*60}")
        display(span)
    else:
        print("DR_Time could not be parsed as a timestamp; temporal tables skipped.")
else:
    print("DR_Time column not found; temporal tables skipped.")


## Step 2 -- PCA and t-SNE visualization


In [ ]:
# PCA
pca   = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_scaled)
var_exp = pca.explained_variance_ratio_

# t-SNE (subsampled if the dataset is large)
MAX_TSNE = 5000
rng_tsne = np.random.default_rng(RANDOM_STATE)  # fixed seed: reproducible t-SNE subsample
idx_tsne = rng_tsne.choice(len(X_scaled), min(MAX_TSNE, len(X_scaled)), replace=False)
# Compatibility with sklearn < 1.4 (n_iter) and >= 1.4 (max_iter)
import sklearn
sk_version = tuple(int(x) for x in sklearn.__version__.split(".")[:2])
tsne_params = dict(n_components=2, perplexity=40, random_state=RANDOM_STATE,
                   learning_rate="auto", init="pca")
if sk_version >= (1, 4):
    tsne_params["max_iter"] = 1000
else:
    tsne_params["n_iter"] = 1000
tsne  = TSNE(**tsne_params)
X_tsne = tsne.fit_transform(X_scaled[idx_tsne])
y_tsne = y[idx_tsne]

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

for ax, X_proj, y_proj, title, xl, yl in [
    (axes[0], X_pca,  y,      
     f"PCA -- explained variance: {sum(var_exp)*100:.1f}%",
     f"PC1 ({var_exp[0]*100:.1f}%)", f"PC2 ({var_exp[1]*100:.1f}%)"),
    (axes[1], X_tsne, y_tsne,
     f"t-SNE (n={len(idx_tsne)})",
     "t-SNE 1", "t-SNE 2"),
]:
    for cls_id, cls_name in enumerate(le.classes_):
        mask = y_proj == cls_id
        ax.scatter(X_proj[mask, 0], X_proj[mask, 1],
                   c=PALETTE.get(cls_name, "#888780"),
                   alpha=0.45, s=10, label=cls_name, rasterized=True)
    
    ax.set_xlabel(xl); ax.set_ylabel(yl); ax.set_title(title)
    ax.legend(fontsize=8, markerscale=2.5,
              loc="upper right", framealpha=0.9)

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/Fig14.pdf", bbox_inches="tight")
plt.savefig(f"{OUTPUT_DIR}/Fig14.png", bbox_inches="tight", dpi=300)
plt.show()

# Silhouette score (class separability)
sil = silhouette_score(X_scaled, y, sample_size=5000, random_state=RANDOM_STATE)
print(f"Silhouette score (class separability): {sil:.3f}")
print("  -> Close to 1 = well-separated classes | Close to 0 = overlap")


### Cluster quality metrics on each representation (addresses reviewer concern: is the PCA/t-SNE picture a faithful reflection of separability, or a projection artifact?)

In [ ]:
# Complements the single full-space silhouette score above with the same
# metric computed on the PCA and t-SNE projections themselves, plus the
# Davies-Bouldin index (lower is better; no fixed upper bound) for all
# three representations. Reported at face value, including when the
# result is unflattering: these metrics assume convex, Euclidean
# clusters, so a low score here does not necessarily contradict a high
# classification accuracy obtained by a non-linear model downstream.
from sklearn.metrics import davies_bouldin_score

cluster_quality_rows = [
    {"Representation": "Full feature space (9D)",
     "Silhouette": round(sil, 3),
     "Davies-Bouldin": round(davies_bouldin_score(X_scaled, y), 3)},
    {"Representation": "PCA (2D)",
     "Silhouette": round(silhouette_score(X_pca, y, sample_size=5000, random_state=RANDOM_STATE), 3),
     "Davies-Bouldin": round(davies_bouldin_score(X_pca, y), 3)},
    {"Representation": "t-SNE (2D)",
     "Silhouette": round(silhouette_score(X_tsne, y_tsne), 3),
     "Davies-Bouldin": round(davies_bouldin_score(X_tsne, y_tsne), 3)},
]
cluster_quality_df = pd.DataFrame(cluster_quality_rows)
cluster_quality_df.to_csv(f"{OUTPUT_DIR}/Table_ClusterQuality.csv", index=False)

print(f"{'='*60}\n  Cluster quality across representations\n{'='*60}")
display(cluster_quality_df)
print("Note: silhouette near/below 0 and a high Davies-Bouldin index indicate "
      "weak geometric (Euclidean) separability -- consistent with, rather "
      "than contradicting, the non-linear classification results below "
      "(Section 5.2.2 of the paper).")


## Step 3 -- Training and model comparison


In [ ]:
# ================================================================
#  MODEL DEFINITIONS BY FAMILY + HYPERPARAMETER GRIDS
# ================================================================

# Train/test split done once -- GridSearchCV uses X_train
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE)

# -- Family 1: Linear methods ------------------------------------------
fam1 = {
    "Logistic Regression": (
        LogisticRegression(random_state=RANDOM_STATE, n_jobs=-1, max_iter=2000),
        {"C": [0.01, 0.1, 1, 10], "solver": ["lbfgs", "saga"],
         "multi_class": ["multinomial"]}
    ),
    "LDA": (
        LinearDiscriminantAnalysis(),
        {"solver": ["svd", "lsqr"], "shrinkage": [None, "auto"]}
    ),
    "QDA": (
        QuadraticDiscriminantAnalysis(),
        {"reg_param": [0.0, 0.1, 0.3, 0.5]}
    ),
    "SGD Classifier": (
        SGDClassifier(random_state=RANDOM_STATE, max_iter=1000, n_jobs=-1),
        {"loss": ["hinge", "log_loss", "modified_huber"],
         "alpha": [0.0001, 0.001, 0.01], "penalty": ["l2", "elasticnet"]}
    ),
}

# -- Family 2: Decision trees -------------------------------------------
fam2 = {
    "Decision Tree": (
        DecisionTreeClassifier(random_state=RANDOM_STATE),
        {"max_depth": [5, 10, 20, None], "min_samples_split": [2, 5, 10],
         "criterion": ["gini", "entropy"]}
    ),
}

# -- Family 3: Ensembles -------------------------------------------------
fam3 = {
    "Random Forest": (
        RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
        {"n_estimators": [300, 600], "max_depth": [10, 20, None],
         "min_samples_split": [2, 5]}
    ),
    "Extra Trees": (
        ExtraTreesClassifier(random_state=RANDOM_STATE, n_jobs=-1),
        {"n_estimators": [300, 600], "max_depth": [10, 20, None]}
    ),
    "AdaBoost": (
        AdaBoostClassifier(random_state=RANDOM_STATE),
        {"n_estimators": [100, 200, 400], "learning_rate": [0.5, 1.0, 1.5]}
    ),
    "Gradient Boosting": (
        GradientBoostingClassifier(random_state=RANDOM_STATE),
        {"n_estimators": [200, 400], "max_depth": [3, 5, 8],
         "learning_rate": [0.05, 0.1, 0.2]}
    ),
}

# XGBoost / LightGBM / CatBoost
if HAS_XGB:
    fam3["XGBoost"] = (
        XGBClassifier(random_state=RANDOM_STATE, verbosity=0,
                      use_label_encoder=False, eval_metric="mlogloss", n_jobs=-1),
        # max_depth capped at 10 (rather than up to 20) given only 10 input
        # features; subsample/colsample_bytree/reg_lambda added as explicit
        # regularization terms to reduce overfitting risk at higher depths.
        {"n_estimators": [300, 600], "max_depth": [4, 6, 10],
         "learning_rate": [0.05, 0.1],
         "subsample": [0.8, 1.0],
         "colsample_bytree": [0.8, 1.0],
         "reg_lambda": [1, 5]}
    )
if HAS_LGB:
    fam3["LightGBM"] = (
        LGBMClassifier(random_state=RANDOM_STATE, verbose=-1, n_jobs=-1),
        {"n_estimators": [300, 600], "max_depth": [6, 12, 20],
         "learning_rate": [0.05, 0.1]}
    )
if HAS_CAT:
    fam3["CatBoost"] = (
        CatBoostClassifier(random_state=RANDOM_STATE, verbose=0),
        {"iterations": [300, 600], "depth": [6, 10],
         "learning_rate": [0.05, 0.1]}
    )

# -- Family 4: Kernel methods --------------------------------------------
fam4 = {
    "SVM (RBF)": (
        SVC(kernel="rbf", probability=True, random_state=RANDOM_STATE),
        {"C": [1, 10, 100], "gamma": ["scale", "auto", 0.01]}
    ),
    "SVM (Poly)": (
        SVC(kernel="poly", probability=True, random_state=RANDOM_STATE),
        {"C": [1, 10], "degree": [2, 3], "gamma": ["scale"]}
    ),
    "Nu-SVC": (
        NuSVC(probability=True, random_state=RANDOM_STATE),
        {"nu": [0.3, 0.5, 0.7], "kernel": ["rbf", "poly"]}
    ),
}

# -- Family 5: Instance-based --------------------------------------------
fam5 = {
    "KNN": (
        KNeighborsClassifier(n_jobs=-1),
        {"n_neighbors": [3, 5, 10, 15, 25],
         "metric": ["euclidean", "manhattan", "minkowski"],
         "weights": ["uniform", "distance"]}
    ),
}

# -- Family 6: Bayesian ---------------------------------------------------
fam6 = {
    "Gaussian NB": (
        GaussianNB(),
        {"var_smoothing": [1e-9, 1e-7, 1e-5, 1e-3]}
    ),
}

# -- Family 7: Neural networks --------------------------------------------
fam7 = {
    "MLP": (
        MLPClassifier(random_state=RANDOM_STATE, max_iter=500),
        {"hidden_layer_sizes": [(64,), (128, 64), (256, 128, 64)],
         "activation": ["relu", "tanh"],
         "alpha": [0.0001, 0.001],
         "learning_rate_init": [0.001, 0.01]}
    ),
}

# -- Grouping by family ----------------------------------------------------
ALL_FAMILIES = {
    "Linear":              fam1,
    "Trees":               fam2,
    "Ensembles":           fam3,
    "Kernel (SVM)":        fam4,
    "Instance-based":      fam5,
    "Bayesian":            fam6,
    "Neural Networks":     fam7,
}

print(f"{'Family':22} | {'Models':>6}")
print("-" * 35)
total = 0
for fam, models in ALL_FAMILIES.items():
    print(f"  {fam:<20} | {len(models):>4}")
    total += len(models)
print(f"{'-'*35}")
print(f"  {'TOTAL':<20} | {total:>4} classifiers")


In [ ]:
# ================================================================
#  GRIDSEARCHCV PER FAMILY + CROSS-VALIDATION
#  Estimated duration: 30-90 min depending on dataset size
# ================================================================

cv_inner  = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
cv_outer  = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)

results      = {}   # {model_name: {acc_mean, acc_std, f1_mean, f1_std, best_params, family}}
best_models  = {}   # {model_name: fitted_model}

PALETTE_FAMILIES = {
    "Linear":            "#378ADD",
    "Trees":             "#BA7517",
    "Ensembles":         "#1D9E75",
    "Kernel (SVM)":      "#D85A30",
    "Instance-based":    "#7F77DD",
    "Bayesian":          "#D4537E",
    "Neural Networks":   "#888780",
}

t_total_start = time.time()

for family_name, family_models in ALL_FAMILIES.items():
    print(f"\n{'='*55}")
    print(f"  Family: {family_name}")
    print(f"{'='*55}")

    for name, (base_model, param_grid) in family_models.items():
        t0 = time.time()
        try:
            # GridSearchCV on the training set only (inner CV = 3-fold)
            gs = GridSearchCV(base_model, param_grid,
                              cv=cv_inner, scoring="f1_macro",
                              n_jobs=-1, refit=True, verbose=0)
            gs.fit(X_train, y_train)
            best_est = gs.best_estimator_

            # Outer CV (5-fold) with the best estimator
            acc = cross_val_score(best_est, X_scaled, y,
                                  cv=cv_outer, scoring="accuracy", n_jobs=-1)
            f1  = cross_val_score(best_est, X_scaled, y,
                                  cv=cv_outer, scoring="f1_macro",  n_jobs=-1)

            results[name] = {
                "acc_mean": acc.mean(), "acc_std": acc.std(),
                "f1_mean":  f1.mean(),  "f1_std":  f1.std(),
                "best_params": gs.best_params_,
                "family": family_name,
                "time_s": round(time.time() - t0, 1),
            }
            best_models[name] = gs.best_estimator_

            print(f"  [ok] {name:<22} Acc={acc.mean():.3f}+/-{acc.std():.3f}  "
                  f"F1={f1.mean():.3f}+/-{f1.std():.3f}  "
                  f"[{time.time()-t0:.0f}s]  "
                  f"params={gs.best_params_}")

        except Exception as e:
            print(f"  [x] {name:<22} ERROR: {e}")

total_time = time.time() - t_total_start
print(f"\n{'='*55}")
print(f"  Finished in {total_time/60:.1f} min | {len(results)} models evaluated")
print(f"{'='*55}")

# Save results
rows = []
for name, r in results.items():
    rows.append({
        "Family":     r["family"],
        "Classifier": name,
        "Accuracy":   round(r["acc_mean"], 4),
        "Acc_std":    round(r["acc_std"],  4),
        "F1_macro":   round(r["f1_mean"],  4),
        "F1_std":     round(r["f1_std"],   4),
        "Time (s)":   r["time_s"],
        "Best params": str(r["best_params"]),
    })
results_df = pd.DataFrame(rows).sort_values("F1_macro", ascending=False)
results_df.to_csv(f"{OUTPUT_DIR}/Table8_cv_results.csv", index=False)

best_name = max(results, key=lambda n: results[n]["f1_mean"])
print(f"\n-> Best overall model: {best_name}  "
      f"(F1 = {results[best_name]['f1_mean']:.3f})")
display(results_df[["Family","Classifier","Accuracy","F1_macro","Time (s)"]].head(10))


## Step 3a -- Statistical significance between top-performing classifiers

In [ ]:
# The top ensemble methods cluster within a narrow F1-macro range (see
# results_df above). A paired t-test on the per-fold F1-macro scores from
# the same cv_outer split checks whether the best model is genuinely
# distinguishable from its closest competitors, rather than comparing
# only the mean +/- std already printed.
from scipy.stats import ttest_rel

TOP_N_FOR_SIGTEST = 6
top_names = (results_df[results_df["Family"] != "Baseline"]
             .sort_values("F1_macro", ascending=False)
             .head(TOP_N_FOR_SIGTEST)["Classifier"].tolist())

fold_f1 = {name: cross_val_score(best_models[name], X_scaled, y, cv=cv_outer,
                                  scoring="f1_macro", n_jobs=-1)
           for name in top_names}

ref_name = top_names[0]
sig_rows = []
for name in top_names[1:]:
    t_stat, p_val = ttest_rel(fold_f1[ref_name], fold_f1[name])
    sig_rows.append({
        "Comparison": f"{ref_name} vs. {name}",
        "Mean_F1_diff": round(fold_f1[ref_name].mean() - fold_f1[name].mean(), 4),
        "p_value": round(p_val, 4),
        "Significant_at_0.05": p_val < 0.05,
    })

sig_df = pd.DataFrame(sig_rows)
sig_df.to_csv(f"{OUTPUT_DIR}/Table_ModelSignificance.csv", index=False)

print(f"{'='*60}\n  Paired t-test: {ref_name} vs. next {len(top_names)-1} closest models\n{'='*60}")
for _, r in sig_df.iterrows():
    verdict = "SIGNIFICANT" if r["Significant_at_0.05"] else "not significant"
    print(f"  {r['Comparison']:<32} diff={r['Mean_F1_diff']:+.4f}  p={r['p_value']:.4f}  -> {verdict}")
display(sig_df)


## Step 3b -- Baselines and group-aware validation


In [ ]:
# -- Simple baselines --
# Majority-class baseline: always predicts the most frequent class.
# RSSI-only baseline: a shallow decision tree using RSSI alone, representing
# a simple threshold-based classification rule.
from sklearn.dummy import DummyClassifier

baseline_results = {}

# Majority class
dummy = DummyClassifier(strategy="most_frequent", random_state=RANDOM_STATE)
acc = cross_val_score(dummy, X_scaled, y, cv=cv_outer, scoring="accuracy", n_jobs=-1)
f1  = cross_val_score(dummy, X_scaled, y, cv=cv_outer, scoring="f1_macro",  n_jobs=-1)
baseline_results["Majority class"] = {
    "acc_mean": acc.mean(), "acc_std": acc.std(),
    "f1_mean":  f1.mean(),  "f1_std":  f1.std(),
    "family": "Baseline",
}

# RSSI-only shallow tree (simple threshold-based rule)
rssi_idx = FEATURE_NAMES.index("rssi")
X_rssi_only = X_scaled[:, [rssi_idx]]
rssi_tree = DecisionTreeClassifier(max_depth=3, random_state=RANDOM_STATE)
acc = cross_val_score(rssi_tree, X_rssi_only, y, cv=cv_outer, scoring="accuracy", n_jobs=-1)
f1  = cross_val_score(rssi_tree, X_rssi_only, y, cv=cv_outer, scoring="f1_macro",  n_jobs=-1)
baseline_results["RSSI-only (threshold)"] = {
    "acc_mean": acc.mean(), "acc_std": acc.std(),
    "f1_mean":  f1.mean(),  "f1_std":  f1.std(),
    "family": "Baseline",
}

print(f"{'Baseline':<24} {'Accuracy':>12} {'F1-macro':>12}")
print("-" * 50)
for name, r in baseline_results.items():
    print(f"{name:<24} {r['acc_mean']:.3f}+/-{r['acc_std']:.3f}  "
          f"{r['f1_mean']:.3f}+/-{r['f1_std']:.3f}")
print(f"\nBest full-feature model ({best_name}): "
      f"Accuracy={results[best_name]['acc_mean']:.3f}, "
      f"F1={results[best_name]['f1_mean']:.3f}")

# Append to results_df so baselines appear alongside the full comparison
baseline_rows = []
for name, r in baseline_results.items():
    baseline_rows.append({
        "Family": r["family"], "Classifier": name,
        "Accuracy": round(r["acc_mean"], 4), "Acc_std": round(r["acc_std"], 4),
        "F1_macro": round(r["f1_mean"], 4), "F1_std": round(r["f1_std"], 4),
        "Time (s)": None, "Best params": "",
    })
results_df = pd.concat([results_df, pd.DataFrame(baseline_rows)], ignore_index=True)
results_df.to_csv(f"{OUTPUT_DIR}/Table8_cv_results.csv", index=False)


In [ ]:
# -- Group-aware cross-validation (GroupKFold by physical node) --
# Packets transmitted by the same node are temporally correlated (fixed
# 30s interval, shared propagation path), so a standard stratified k-fold
# can place packets from the same node in both training and validation
# folds. GroupKFold instead ensures all packets from a given node stay
# together in the same fold, giving a stricter estimate of generalization
# to unseen nodes.
from sklearn.model_selection import GroupKFold

# Build a node-group identifier unique across environments (Device_ID
# alone repeats, e.g. "noeud1", across different environments/files).
node_group = (df_all["environment"].astype(str) + "_" +
              df_all["Device_ID"].astype(str)).values

n_unique_nodes = len(np.unique(node_group))
group_cv = GroupKFold(n_splits=min(CV_FOLDS, n_unique_nodes))

best_est_for_group_cv = best_models[best_name]

acc_group = cross_val_score(best_est_for_group_cv, X_scaled, y,
                            groups=node_group, cv=group_cv,
                            scoring="accuracy", n_jobs=-1)
f1_group  = cross_val_score(best_est_for_group_cv, X_scaled, y,
                            groups=node_group, cv=group_cv,
                            scoring="f1_macro", n_jobs=-1)

print(f"{'='*60}")
print(f"  Group-aware validation (GroupKFold by node) -- {best_name}")
print(f"  {n_unique_nodes} unique nodes across all environments")
print(f"{'='*60}")
print(f"  Standard stratified CV : Accuracy={results[best_name]['acc_mean']:.3f}+/-{results[best_name]['acc_std']:.3f}  "
      f"F1={results[best_name]['f1_mean']:.3f}+/-{results[best_name]['f1_std']:.3f}")
print(f"  GroupKFold (by node)   : Accuracy={acc_group.mean():.3f}+/-{acc_group.std():.3f}  "
      f"F1={f1_group.mean():.3f}+/-{f1_group.std():.3f}")
print(f"{'='*60}")

# Save as a table for the manuscript (Table: Standard vs Group-Aware CV)
groupcv_df = pd.DataFrame([
    {"Validation protocol": "Stratified 5-fold (packet-level)",
     "Accuracy": round(results[best_name]['acc_mean'], 4),
     "Accuracy_std": round(results[best_name]['acc_std'], 4),
     "F1_macro": round(results[best_name]['f1_mean'], 4),
     "F1_macro_std": round(results[best_name]['f1_std'], 4)},
    {"Validation protocol": f"GroupKFold (node-level, {n_unique_nodes} nodes)",
     "Accuracy": round(acc_group.mean(), 4),
     "Accuracy_std": round(acc_group.std(), 4),
     "F1_macro": round(f1_group.mean(), 4),
     "F1_macro_std": round(f1_group.std(), 4)},
])
groupcv_df.to_csv(f"{OUTPUT_DIR}/Table_GroupCV.csv", index=False)
display(groupcv_df)


## Step 4 -- Detailed evaluation of the best model


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE)

best_model = best_models[best_name]

# For XGBoost, refit the final model with early stopping on a validation
# split carved out of the training set, to avoid relying solely on the
# fixed n_estimators selected by GridSearchCV (which does not use early
# stopping internally, for consistency with the other 17 classifiers).
if HAS_XGB and isinstance(best_model, XGBClassifier):
    X_tr, X_val, y_tr, y_val = train_test_split(
        X_train, y_train, test_size=0.15, stratify=y_train,
        random_state=RANDOM_STATE)
    best_model.set_params(n_estimators=2000)
    try:
        # xgboost >= 2.0 API
        best_model.set_params(early_stopping_rounds=30)
        best_model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
    except TypeError:
        # xgboost < 2.0 API (early_stopping_rounds is a fit() argument,
        # not a constructor/set_params argument)
        best_model.set_params(early_stopping_rounds=None)
        best_model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)],
                       early_stopping_rounds=30, verbose=False)
    n_trees_used = getattr(best_model, "best_iteration", None)
    if n_trees_used is not None:
        print(f"Early stopping: best_iteration = {n_trees_used + 1} "
              f"(out of 2000 max)")

    # IMPORTANT: clear early_stopping_rounds before refitting on the full
    # training set below -- otherwise XGBoost raises "Must have at least
    # 1 validation dataset for early stopping" since no eval_set is
    # passed to this second fit.
    best_model.set_params(early_stopping_rounds=None)
    if n_trees_used is not None:
        best_model.set_params(n_estimators=n_trees_used + 1)
    best_model.fit(X_train, y_train)
else:
    best_model.fit(X_train, y_train)

y_pred = best_model.predict(X_test)

print(f"{'='*60}")
print(f"  {best_name} -- evaluation on 25% held-out test set")
print(f"{'='*60}")
print(classification_report(y_test, y_pred, target_names=le.classes_,
                             digits=3))

# Save as a table for the manuscript
report_dict = classification_report(y_test, y_pred, target_names=le.classes_,
                                     digits=3, output_dict=True)
report_df = pd.DataFrame(report_dict).transpose().round(4)
report_df.to_csv(f"{OUTPUT_DIR}/Table_ClassificationReport.csv")


In [ ]:
# -- Confusion Matrix --
n_classes = len(le.classes_)
fig_w = max(7, n_classes * 1.2)
fig, ax = plt.subplots(figsize=(fig_w, fig_w * 0.8))
cm   = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=le.classes_)
disp.plot(ax=ax, cmap="Blues", colorbar=False, values_format="d")
ax.set_title(f"Confusion Matrix -- {best_name} (n={cm.sum()} test packets)")
plt.xticks(rotation=35, ha="right", fontsize=9)
plt.yticks(fontsize=9)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/Fig16.png", bbox_inches="tight", dpi=300)
plt.savefig(f"{OUTPUT_DIR}/Fig16.pdf", bbox_inches="tight")
plt.show()

# Most frequent confusions
print("\nMain confusions:")
for i in range(n_classes):
    for j in range(n_classes):
        if i != j and cm[i, j] > 0:
            pct = cm[i, j] / cm[i].sum() * 100
            print(f"  {le.classes_[i]:<22} -> {le.classes_[j]:<22} : "
                  f"{cm[i,j]:>4} packets ({pct:.1f}%)")

# Save the confusion matrix itself as a table
cm_df = pd.DataFrame(cm, index=le.classes_, columns=le.classes_)
cm_df.to_csv(f"{OUTPUT_DIR}/Table_ConfusionMatrix.csv")


In [ ]:
# -- Same-environment vs cross-environment misclassifications --
# Only informative in 7-class mode: breaks down confusion matrix errors
# into "same physical site, different LOS/NLOS/scenario" vs "genuinely
# different environment", to check whether the classifier is primarily
# distinguishing LOS/NLOS status rather than the environment itself.

if CLASSIFICATION_MODE == "7-class":
    def base_environment(label):
        return label.split("_")[0]

    same_env_errors = 0
    cross_env_errors = 0
    for i in range(n_classes):
        for j in range(n_classes):
            if i == j:
                continue
            if base_environment(le.classes_[i]) == base_environment(le.classes_[j]):
                same_env_errors += cm[i, j]
            else:
                cross_env_errors += cm[i, j]

    total_errors = same_env_errors + cross_env_errors
    if total_errors > 0:
        print(f"{'='*60}")
        print("  Error breakdown (7-class mode)")
        print(f"{'='*60}")
        print(f"  Same-environment errors (e.g. LOS <-> NLOS within one site): "
              f"{same_env_errors} ({same_env_errors/total_errors*100:.1f}% of all errors)")
        print(f"  Cross-environment errors (different physical sites): "
              f"{cross_env_errors} ({cross_env_errors/total_errors*100:.1f}% of all errors)")
        print(f"{'='*60}")
    else:
        print("No misclassifications to break down.")


In [ ]:
# -- ROC Curves, one-vs-rest --
y_test_bin = label_binarize(y_test, classes=range(n_classes))
y_score    = best_model.predict_proba(X_test)

fig, ax = plt.subplots(figsize=(8, 6))
roc_colors = list(PALETTE.values())

auc_scores = {}
for i, cls in enumerate(le.classes_):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_score[:, i])
    roc_auc = auc(fpr, tpr)
    auc_scores[cls] = roc_auc
    ax.plot(fpr, tpr, color=roc_colors[i % len(roc_colors)],
            linewidth=2, label=f"{cls}  (AUC={roc_auc:.3f})")

ax.plot([0,1],[0,1], "--", color="#888780", linewidth=1, label="Random")
ax.set_xlabel("False Positive Rate (FPR)")
ax.set_ylabel("True Positive Rate (TPR)")
ax.legend(fontsize=8, loc="lower right")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/Fig17.pdf", bbox_inches="tight")
plt.savefig(f"{OUTPUT_DIR}/Fig17.png", bbox_inches="tight", dpi=300)
plt.show()

print("AUC per environment:")
for cls, a in sorted(auc_scores.items(), key=lambda x: -x[1]):
    print(f"  {cls:<25} AUC = {a:.4f}")

# Save AUC scores as a table
auc_df = pd.DataFrame(sorted(auc_scores.items(), key=lambda x: -x[1]),
                      columns=["Environment", "AUC"]).round(4)
auc_df.to_csv(f"{OUTPUT_DIR}/Table_AUC.csv", index=False)


## Step 5 -- Which radio metrics discriminate best?


In [ ]:
# -- Permutation importance (model-agnostic) --
perm = permutation_importance(best_model, X_test, y_test,
                               n_repeats=30, random_state=RANDOM_STATE, n_jobs=-1)
imp_df = pd.DataFrame({
    "feature":    FEATURE_NAMES,
    "importance": perm.importances_mean,
    "std":        perm.importances_std,
}).sort_values("importance", ascending=False)

fig, ax = plt.subplots(figsize=(8, 5))
colors_bar = ["#D85A30" if v >= imp_df["importance"].quantile(0.66)
              else ("#378ADD" if v >= imp_df["importance"].quantile(0.33)
              else "#B5D4F4")
              for v in imp_df.sort_values("importance")["importance"]]
ax.barh(imp_df.sort_values("importance")["feature"],
        imp_df.sort_values("importance")["importance"],
        xerr=imp_df.sort_values("importance")["std"],
        color=colors_bar, capsize=3, edgecolor="#444441", linewidth=0.4)
ax.axvline(0, color="#888780", linewidth=0.8, linestyle="--")
ax.set_xlabel("Permutation importance")

# Color legend
from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(color="#D85A30", label="Top tier"),
    Patch(color="#378ADD", label="Mid tier"),
    Patch(color="#B5D4F4", label="Low tier"),
], fontsize=8, loc="lower right")

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/Fig18.pdf", bbox_inches="tight")
plt.savefig(f"{OUTPUT_DIR}/Fig18.png", bbox_inches="tight", dpi=300)
plt.show()

imp_df.round(4).to_csv(f"{OUTPUT_DIR}/Table9_feature_importance.csv", index=False)
display(imp_df.reset_index(drop=True))


## Step 5b -- Feature ablation study (drop-one-feature impact on F1)

In [ ]:
# Complements permutation importance (which can under-state the effect
# of correlated features) by retraining the best model's architecture
# with each feature entirely removed, rather than merely permuted.
from sklearn.base import clone

baseline_f1 = results[best_name]["f1_mean"]

ablation_rows = [{"Removed_feature": "(none -- full model)",
                  "F1_macro": round(baseline_f1, 4), "F1_drop": 0.0}]

for i, feat in enumerate(FEATURE_NAMES):
    cols_kept = [j for j in range(len(FEATURE_NAMES)) if j != i]
    X_ablated = X_scaled[:, cols_kept]
    model_ablated = clone(best_models[best_name])
    f1_scores = cross_val_score(model_ablated, X_ablated, y, cv=cv_outer,
                                scoring="f1_macro", n_jobs=-1)
    ablation_rows.append({
        "Removed_feature": feat,
        "F1_macro": round(f1_scores.mean(), 4),
        "F1_drop": round(baseline_f1 - f1_scores.mean(), 4),
    })
    print(f"  Without '{feat}': F1={f1_scores.mean():.4f} "
          f"(drop of {baseline_f1 - f1_scores.mean():+.4f})")

ablation_df = pd.DataFrame(ablation_rows).sort_values("F1_drop", ascending=False)
ablation_df.to_csv(f"{OUTPUT_DIR}/Table_Ablation.csv", index=False)

fig, ax = plt.subplots(figsize=(8, 5))
plot_abl = ablation_df[ablation_df["Removed_feature"] != "(none -- full model)"]
ax.barh(plot_abl["Removed_feature"], plot_abl["F1_drop"],
        color="#D85A30", edgecolor="#444441", linewidth=0.4)
ax.axvline(0, color="#888780", linewidth=0.8, linestyle="--")
ax.set_xlabel(f"F1-macro drop when feature is removed (baseline F1={baseline_f1:.4f})")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/Fig_Ablation.pdf", bbox_inches="tight")
plt.savefig(f"{OUTPUT_DIR}/Fig_Ablation.png", bbox_inches="tight", dpi=300)
plt.show()
display(ablation_df)


## Step 5c -- SHAP analysis (complementing permutation importance)

In [ ]:
# Optional: SHAP decomposes each individual prediction into per-feature
# contributions and is more robust to feature correlation than
# permutation importance, at higher computational cost. Requires
# `pip install shap`; skipped gracefully if unavailable.
try:
    import shap
    HAS_SHAP = True
except ImportError:
    HAS_SHAP = False
    print("[!] shap not installed -- run `pip install shap` to enable this cell.")

if HAS_SHAP:
    SHAP_SAMPLE_SIZE = min(2000, len(X_test))
    X_shap_sample = X_test[:SHAP_SAMPLE_SIZE]

    tree_based = any(k in type(best_model).__name__
                     for k in ["XGB", "LGBM", "CatBoost", "Forest", "Trees", "GradientBoosting"])

    if tree_based:
        explainer = shap.TreeExplainer(best_model)
        shap_values = explainer.shap_values(X_shap_sample)
    else:
        background = shap.sample(X_train, 100, random_state=RANDOM_STATE)
        explainer = shap.KernelExplainer(best_model.predict_proba, background)
        shap_values = explainer.shap_values(X_shap_sample, nsamples=100)

    mean_abs_shap = (np.mean([np.abs(sv).mean(axis=0) for sv in shap_values], axis=0)
                     if isinstance(shap_values, list)
                     else np.abs(shap_values).mean(axis=0))

    shap_importance_df = pd.DataFrame({
        "feature": FEATURE_NAMES, "mean_abs_shap": mean_abs_shap,
    }).sort_values("mean_abs_shap", ascending=False)
    shap_importance_df.to_csv(f"{OUTPUT_DIR}/Table_SHAP_Importance.csv", index=False)
    display(shap_importance_df)

    plt.figure(figsize=(8, 5))
    if isinstance(shap_values, list):
        shap.summary_plot(shap_values, X_shap_sample, feature_names=FEATURE_NAMES,
                          plot_type="bar", class_names=le.classes_, show=False)
    else:
        shap.summary_plot(shap_values, X_shap_sample, feature_names=FEATURE_NAMES,
                          plot_type="bar", show=False)
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/Fig_SHAP_Summary.pdf", bbox_inches="tight")
    plt.savefig(f"{OUTPUT_DIR}/Fig_SHAP_Summary.png", bbox_inches="tight", dpi=300)
    plt.show()


## Step 6 -- Average feature profile per environment


In [ ]:
# Average profile per environment (helps interpret the important features)
profile = (pd.DataFrame(X_raw.values, columns=FEATURE_NAMES)
           .assign(environment=df_all["environment"].values)
           .groupby("environment")[FEATURE_NAMES]
           .mean().round(3))

# Radar chart
from matplotlib.patches import FancyArrowPatch

top_features = imp_df.head(6)["feature"].tolist()
angles = np.linspace(0, 2*np.pi, len(top_features), endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
for env, color in PALETTE.items():
    if env not in profile.index:
        continue
    vals = profile.loc[env, top_features].tolist()
    # Normalize 0-1 for the radar chart
    col_min = profile[top_features].min()
    col_max = profile[top_features].max()
    vals_norm = [(v - col_min[f]) / max(col_max[f] - col_min[f], 1e-9)
                 for v, f in zip(vals, top_features)]
    vals_norm += vals_norm[:1]
    ax.plot(angles, vals_norm, color=color, linewidth=1.8, label=env)
    ax.fill(angles, vals_norm, color=color, alpha=0.08)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(top_features, fontsize=9)
ax.legend(loc="upper right", bbox_to_anchor=(1.35, 1.15), fontsize=8)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/Fig19.pdf", bbox_inches="tight")
plt.savefig(f"{OUTPUT_DIR}/Fig19.png", bbox_inches="tight", dpi=300)
plt.show()


## Step 7 -- Predict the environment of a new packet


In [ ]:
def predict_environment(rssi, snr, sf=7, freq_mhz=868.1,
                        airtime_ms=46.3, freq_offset=-400,
                        rssi_delta=0, conditions_1=np.nan):
    """Predict the LoRa environment of a packet from its radio metrics."""
    feats = [rssi, snr, sf, freq_mhz, airtime_ms, freq_offset, rssi_delta,
             rssi / (snr + 20), int(snr > 0)]
    if "conditions_1" in FEATURE_NAMES:
        feats.append(conditions_1)

    feats_scaled = scaler.transform([feats])
    pred_id      = best_model.predict(feats_scaled)[0]
    proba        = best_model.predict_proba(feats_scaled)[0]
    pred_label   = le.inverse_transform([pred_id])[0]

    print(f"  RSSI={rssi} dBm | SNR={snr} dB | SF{sf} | {freq_mhz} MHz")
    print(f"  -> Predicted environment: {pred_label}  "
          f"(confidence: {proba.max()*100:.1f}%)")
    top3 = sorted(zip(le.classes_, proba), key=lambda x: -x[1])[:3]
    print(f"  Top 3: " + " | ".join(f"{c}: {p:.2f}" for c, p in top3))
    print()
    return pred_label, proba

# Representative examples for each environment
print("Prediction examples\n" + "-"*50)
predict_environment(rssi=-50, snr=13.8, sf=7,  freq_mhz=868.3)  # Residential LOS
predict_environment(rssi=-62, snr=11.5, sf=7,  freq_mhz=867.5)  # Residential NLOS
predict_environment(rssi=-57, snr=14.0, sf=7,  freq_mhz=868.1)  # Laboratory LOS
predict_environment(rssi=-100, snr=2.5, sf=12, freq_mhz=867.1)  # Multi-floor
predict_environment(rssi=-82, snr=-3.0, sf=10, freq_mhz=867.9)  # Industrial


## Step 8 -- Inference latency measurement


In [ ]:
# ================================================================
#  INFERENCE LATENCY MEASUREMENT
#  Times real prediction latency on YOUR machine
#  -> value cited in the paper, with measured source
# ================================================================
import time
import numpy as np

N_RUNS = 20000  # number of predictions for a stable measurement

# A single packet (normalized feature vector)
single_packet = X_test[0:1]

# -- Warm-up (JIT of the model) --------------------------------
for _ in range(100):
    best_model.predict(single_packet)

# -- Measurement over N_RUNS individual predictions -------------
latencies = []
for _ in range(N_RUNS):
    t0 = time.perf_counter()
    best_model.predict(single_packet)
    latencies.append((time.perf_counter() - t0) * 1000)  # ms

latencies = np.array(latencies)

print(f"{'='*55}")
print(f"  Inference latency -- {best_name}")
print(f"  (measured over {N_RUNS:,} individual predictions)")
print(f"{'='*55}")
print(f"  Mean         : {latencies.mean():.4f} ms")
print(f"  Median       : {np.median(latencies):.4f} ms")
print(f"  P95          : {np.percentile(latencies, 95):.4f} ms")
print(f"  P99          : {np.percentile(latencies, 99):.4f} ms")
print(f"  Min          : {latencies.min():.4f} ms")
print(f"  Max          : {latencies.max():.4f} ms")
print(f"{'='*55}")

# -- Batch latency (100 packets at once) -------------------------
batch = X_test[:100]
batch_latencies = []
for _ in range(1000):
    t0 = time.perf_counter()
    best_model.predict(batch)
    batch_latencies.append((time.perf_counter() - t0) * 1000)

batch_latencies = np.array(batch_latencies)
print(f"  Batch latency (100 packets):")
print(f"  Mean         : {batch_latencies.mean():.4f} ms")
print(f"  Per packet   : {batch_latencies.mean()/100:.6f} ms")
print(f"{'='*55}")

# -- Ready-to-paste sentence for the paper -----------------------
med = np.median(latencies)
p99 = np.percentile(latencies, 99)
print(f"\n  Sentence for the paper (adapt as needed):")
print(f"  \"The inference latency of the {best_name} classifier,")
print(f"  measured over {N_RUNS:,} individual predictions on a")
print(f"  commodity machine, yields a median of {med:.3f} ms")
print(f"  (99th percentile: {p99:.3f} ms) per packet,")
print(f"  confirming its suitability for real-time deployment\"")

# Save latency statistics as a table
latency_df = pd.DataFrame([{
    "Mean_ms": round(latencies.mean(), 4),
    "Median_ms": round(np.median(latencies), 4),
    "P95_ms": round(np.percentile(latencies, 95), 4),
    "P99_ms": round(np.percentile(latencies, 99), 4),
    "Min_ms": round(latencies.min(), 4),
    "Max_ms": round(latencies.max(), 4),
    "Batch100_mean_ms": round(batch_latencies.mean(), 4),
    "Batch100_per_packet_ms": round(batch_latencies.mean() / 100, 6),
    "N_runs": N_RUNS,
}])
latency_df.to_csv(f"{OUTPUT_DIR}/Table_InferenceLatency.csv", index=False)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# -- Data --------------------------------------------------------
med  = np.median(latencies)
p95  = np.percentile(latencies, 95)
p99  = np.percentile(latencies, 99)
mean = latencies.mean()

fig = plt.figure(figsize=(10, 4))
gs  = gridspec.GridSpec(1, 2, width_ratios=[2, 1], wspace=0.35)

# -- Left panel: histogram + KDE ----------------------------------
ax1 = fig.add_subplot(gs[0])
ax1.hist(latencies, bins=80, color="#1D9E75", alpha=0.75,
         edgecolor="white", linewidth=0.3, density=True)
ax1.axvline(med,  color="#D85A30", lw=1.8, linestyle="-",
            label=f"Median = {med:.3f} ms")
ax1.axvline(mean, color="#378ADD", lw=1.5, linestyle="--",
            label=f"Mean = {mean:.3f} ms")
ax1.axvline(p99,  color="#BA7517", lw=1.5, linestyle=":",
            label=f"P99 = {p99:.3f} ms")
ax1.set_xlabel("Inference latency (ms)", fontsize=11)
ax1.set_ylabel("Density", fontsize=11)
ax1.set_title("XGBoost -- Single-packet inference latency\n"
              f"(n = {len(latencies):,} predictions)", fontsize=11)
ax1.legend(fontsize=9)
ax1.set_xlim(0, p99 * 1.5)
ax1.spines["top"].set_visible(False)
ax1.spines["right"].set_visible(False)

# -- Right panel: boxplot -------------------------------------------
ax2 = fig.add_subplot(gs[1])
bp = ax2.boxplot(latencies, vert=True, patch_artist=True,
                 showfliers=False,
                 boxprops=dict(facecolor="#1D9E75", alpha=0.7),
                 medianprops=dict(color="#D85A30", linewidth=2),
                 whiskerprops=dict(color="#555555"),
                 capprops=dict(color="#555555"))
ax2.set_ylabel("Latency (ms)", fontsize=11)
ax2.set_title("Distribution\n(outliers excluded)", fontsize=11)
ax2.set_xticks([])
ax2.spines["top"].set_visible(False)
ax2.spines["right"].set_visible(False)

# Statistical annotations
stats = [
    ("Median", med,  "#D85A30"),
    ("Mean",   mean, "#378ADD"),
    ("P95",    p95,  "#7F77DD"),
    ("P99",    p99,  "#BA7517"),
]
for label, val, color in stats:
    ax2.axhline(val, color=color, lw=1.2, linestyle="--", alpha=0.8)
    ax2.text(1.35, val, f"{label}\n{val:.3f} ms",
             fontsize=7.5, va="center", color=color)

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/Fig20.pdf", bbox_inches="tight")
plt.savefig(f"{OUTPUT_DIR}/Fig20.png", bbox_inches="tight", dpi=300)
plt.show()
print(f"{OUTPUT_DIR}/Fig20.pdf generated")


## Step 8b -- Inference latency across all classifiers

In [ ]:
# The measurement above focuses on the single best model. This extends
# it to all fitted classifiers, so the accuracy/F1 comparison (results_df)
# can be read alongside a computational-cost comparison rather than for
# the best model alone.
N_LATENCY_RUNS = 500
single_packet_ref = X_test[0:1]

latency_all_rows = []
for name, model in best_models.items():
    try:
        for _ in range(20):
            model.predict(single_packet_ref)
        lat = []
        for _ in range(N_LATENCY_RUNS):
            t0 = time.perf_counter()
            model.predict(single_packet_ref)
            lat.append((time.perf_counter() - t0) * 1000)
        lat = np.array(lat)
        latency_all_rows.append({
            "Classifier": name, "Family": results[name]["family"],
            "F1_macro": round(results[name]["f1_mean"], 4),
            "Median_latency_ms": round(np.median(lat), 4),
            "P95_latency_ms": round(np.percentile(lat, 95), 4),
            "Training_time_s": results[name]["time_s"],
        })
    except Exception as e:
        print(f"  [x] {name}: latency measurement failed ({e})")

latency_all_df = pd.DataFrame(latency_all_rows).sort_values("F1_macro", ascending=False)
latency_all_df.to_csv(f"{OUTPUT_DIR}/Table_LatencyAllModels.csv", index=False)
print(f"{'='*60}\n  Inference latency across {len(latency_all_df)} classifiers\n{'='*60}")
display(latency_all_df)

fig, ax = plt.subplots(figsize=(9, 6))
for fam in latency_all_df["Family"].unique():
    mask = latency_all_df["Family"] == fam
    ax.scatter(latency_all_df.loc[mask, "Median_latency_ms"],
               latency_all_df.loc[mask, "F1_macro"],
               label=fam, s=60, edgecolor="#444441", linewidth=0.6,
               color=PALETTE_FAMILIES.get(fam, "#888780"))
for _, row in latency_all_df.iterrows():
    ax.annotate(row["Classifier"], (row["Median_latency_ms"], row["F1_macro"]),
                fontsize=7, xytext=(4, 2), textcoords="offset points")
ax.set_xscale("log")
ax.set_xlabel("Median single-packet inference latency (ms, log scale)")
ax.set_ylabel("F1-macro")
ax.legend(fontsize=8, loc="lower right")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/Fig_LatencyVsF1_AllModels.pdf", bbox_inches="tight")
plt.savefig(f"{OUTPUT_DIR}/Fig_LatencyVsF1_AllModels.png", bbox_inches="tight", dpi=300)
plt.show()


## Summary -- results to include in the paper


In [ ]:
print("=" * 62)
print("  ML RESULTS -- FOR THE PAPER")
print("=" * 62)
print(f"  Total dataset        : {len(df_all)} packets | {len(le.classes_)} classes")
print()
print("  Model comparison (5-fold CV):")
print(f"  {'Model':<22} {'Accuracy':>10} {'F1-macro':>10}")
print("  " + "-"*44)
for name in sorted(results, key=lambda n: -results[n]['f1_mean']):
    r = results[name]
    print(f"  {name:<22} {r['acc_mean']:.3f}+/-{r['acc_std']:.3f}  "
          f"{r['f1_mean']:.3f}+/-{r['f1_std']:.3f}")
print()
print(f"  Best model           : {best_name}")
print(f"  Silhouette score     : {sil:.3f}")
print()
print("  Top 5 discriminant features:")
for _, row in imp_df.head(5).iterrows():
    print(f"    {row['feature']:<22} importance = {row['importance']:.4f}")
print()
print(f"  AUC per environment:")
for cls, a in sorted(auc_scores.items(), key=lambda x: -x[1]):
    print(f"    {cls:<25} {a:.4f}")
print()
print(f"  Figures: {OUTPUT_DIR}/")
print("=" * 62)


In [ ]:
# -- Figure 15: Accuracy vs F1-macro (with +/-1 SD error bars) --
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D

plot_df = results_df.copy()
for c in ("Acc_std", "F1_std"):
    if c not in plot_df.columns:
        plot_df[c] = 0.0
    plot_df[c] = plot_df[c].fillna(0.0)

fig, (ax, axz) = plt.subplots(1, 2, figsize=(15, 7.2),
                              gridspec_kw={"width_ratios": [1.0, 1.15], "wspace": 0.4})

def draw_points(a, df, size):
    for fam, color in PALETTE_FAMILIES.items():
        m = df["Family"] == fam
        if not m.any():
            continue
        a.errorbar(df.loc[m, "Accuracy"], df.loc[m, "F1_macro"],
                   xerr=df.loc[m, "Acc_std"], yerr=df.loc[m, "F1_std"],
                   fmt="none", ecolor=color, elinewidth=1.0, capsize=2, alpha=0.8, zorder=4)
        a.scatter(df.loc[m, "Accuracy"], df.loc[m, "F1_macro"], color=color, s=size,
                  edgecolor="#555555", linewidth=0.8, alpha=0.95, zorder=5)

# ---- (a) all classifiers; labels only for the points outside the zoomed cluster ----
x_cut = plot_df["Accuracy"].max() - 0.075            # accuracy threshold of the top cluster
top = plot_df[plot_df["Accuracy"] >= x_cut].copy()
rest = plot_df[plot_df["Accuracy"] < x_cut]

draw_points(ax, plot_df, 55)
# Alternate labels left/right with a small vertical stagger and a leader line
for i, (_, r) in enumerate(rest.sort_values("F1_macro").iterrows()):
    right = (i % 2 == 0)
    dx = 38 if right else -38
    dy = ((i % 3) - 1) * 13
    ax.annotate(r["Classifier"], (r["Accuracy"], r["F1_macro"]), xytext=(dx, dy),
                textcoords="offset points", fontsize=8, zorder=6,
                ha="left" if right else "right", va="center",
                arrowprops=dict(arrowstyle="-", color="#888888", linewidth=0.6, shrinkA=0, shrinkB=2),
                bbox=dict(boxstyle="round,pad=0.15", facecolor="white",
                          edgecolor=PALETTE_FAMILIES.get(r["Family"], "#888780"),
                          linewidth=0.8, alpha=0.95))

lo = min(plot_df["Accuracy"].min(), plot_df["F1_macro"].min()) - 0.05
hi = max(plot_df["Accuracy"].max(), plot_df["F1_macro"].max()) + 0.04
ax.plot([lo, hi], [lo, hi], linestyle="--", color="#bdbdbd", linewidth=0.8, zorder=1)
x0, x1 = top["Accuracy"].min() - 0.012, top["Accuracy"].max() + 0.012
y0, y1 = top["F1_macro"].min() - 0.012, top["F1_macro"].max() + 0.012
ax.add_patch(plt.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, edgecolor="#444444",
                           linestyle="-", linewidth=1.0, zorder=3))
ax.text(x0, y1 + 0.012, "zoom in (b)", fontsize=8, color="#444444", ha="right")
ax.set_xlim(lo, hi); ax.set_ylim(lo, hi)
ax.set_xlabel("Accuracy", fontsize=11); ax.set_ylabel("F1-macro", fontsize=11)
ax.set_title("(a) All classifiers", fontsize=11, fontweight="bold")
ax.tick_params(labelsize=9)

# ---- (b) zoom on the top cluster, labels in a column with leader lines ----
draw_points(axz, top, 70)
axz.plot([x0 - 1, x1 + 1], [x0 - 1, x1 + 1], linestyle="--", color="#bdbdbd", linewidth=0.8, zorder=1)
axz.set_xlim(x0, x1); axz.set_ylim(y0, y1)
top = top.sort_values("F1_macro", ascending=False).reset_index(drop=True)
ys = np.linspace(y1 - 0.03 * (y1 - y0), y0 + 0.03 * (y1 - y0), len(top))
x_text = x1 + 0.08 * (x1 - x0)
for (_, r), yt in zip(top.iterrows(), ys):
    col = PALETTE_FAMILIES.get(r["Family"], "#888780")
    axz.annotate(r["Classifier"], xy=(r["Accuracy"], r["F1_macro"]), xytext=(x_text, yt),
                 textcoords="data", fontsize=9, annotation_clip=False, zorder=6,
                 arrowprops=dict(arrowstyle="-", color=col, linewidth=0.8, shrinkA=0, shrinkB=3),
                 bbox=dict(boxstyle="round,pad=0.2", facecolor="white", edgecolor=col, linewidth=0.9))
axz.set_xlabel("Accuracy", fontsize=11); axz.set_ylabel("F1-macro", fontsize=11)
axz.set_title("(b) Zoom on the top cluster", fontsize=11, fontweight="bold")
axz.tick_params(labelsize=9)

handles_fam = [mpatches.Patch(color=c, label=f) for f, c in PALETTE_FAMILIES.items()
               if (plot_df["Family"] == f).any()]
handles = handles_fam + [Line2D([0], [0], color="#bdbdbd", linestyle="--", linewidth=0.8, label="F1 = Acc"),
                         Line2D([0], [0], color="#555555", marker="|", linestyle="none", label="$\\pm$1 SD (5-fold CV)")]
ax.legend(handles=handles, loc="upper left", fontsize=8, framealpha=0.95)

plt.savefig(f"{OUTPUT_DIR}/Fig15.pdf", bbox_inches="tight")
plt.savefig(f"{OUTPUT_DIR}/Fig15.png", dpi=300, bbox_inches="tight")
plt.show()
